# 11 — Embedding Regression Comparison

Compares hidden-layer representations across three model variants to measure
whether centroid projection improves linear separability:

- **Model A (06):** No projection — baseline
- **Model B (07):** CentroidProjection during training only (no-op at eval)
- **Model C (09):** CentroidProjection during training AND inference (nearest-centroid assignment)

**Approach:**
1. Train all three models from scratch (same seed, same data, same epochs)
2. Extract full-dimensional hidden_2 activations (64D) for the full test set
3. For B and C, also extract post-projection activations
4. Fit logistic regression on each set of activations
5. Report accuracy + silhouette scores

**Why retrain?** Saved embeddings are 3D UMAP (lossy) and only cover 500 samples.
We need raw 64D activations across the full test set. Same seed = identical results.

In [10]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, silhouette_score

torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__}")
device = torch.device("cpu")

PyTorch 2.10.0


## 1. Load MNIST

In [11]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1)),
])

data_root = os.path.join(PROJECT_ROOT, "data")
full_train_dataset = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    full_train_dataset, [48000, 12000],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, generator=torch.Generator().manual_seed(0))
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 48000, Val: 12000, Test: 10000


## 2. Define All Three Models

In [12]:
PROJ_ALPHA = 0.1
PROJ_BETA = 0.03

# Model A: 06-style MLP (no projection)
class MLP_A(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.ln2 = nn.LayerNorm(64)
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.drop1(torch.relu(self.bn1(self.fc1(x))))
        x = self.drop2(torch.relu(self.ln2(self.fc2(x))))
        x = self.fc3(x)
        return x

# CentroidProjection with optional inference mode
class CentroidProjection(nn.Module):
    def __init__(self, dim, num_classes=10, alpha=0.1, beta=0.05, infer=False):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes
        self.infer = infer
        self.register_buffer('centroids', torch.zeros(num_classes, dim))
        self.register_buffer('ready', torch.tensor(False))

    def update_centroids(self, centroids):
        self.centroids.copy_(centroids)
        self.ready.fill_(True)

    def forward(self, x, labels=None):
        if not self.ready:
            return x
        if labels is None:
            if not self.infer:
                return x
            with torch.no_grad():
                dists = torch.cdist(x, self.centroids)
                labels = dists.argmin(dim=1)

        with torch.no_grad():
            repulsion = torch.zeros_like(self.centroids)
            for c in range(self.num_classes):
                others = [i for i in range(self.num_classes) if i != c]
                diffs = self.centroids[c] - self.centroids[others]
                dists = diffs.norm(dim=1, keepdim=True).clamp(min=1e-6)
                repulsion[c] = (diffs / dists ** 2).sum(dim=0)
            rep_norms = repulsion.norm(dim=1, keepdim=True).clamp(min=1e-6)
            repulsion = repulsion / rep_norms * self.beta
            nudge = repulsion[labels]
            diff = self.centroids[labels] - x
            dist = diff.norm(dim=1, keepdim=True).clamp(min=1e-6)
            mean_dist = dist.mean()
            scale = dist / (mean_dist + 1e-8)
            nudge = nudge + self.alpha * scale * diff

        return x + nudge

# Model B: 07-style (projection during training only)
class MLP_B(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.ln2 = nn.LayerNorm(64)
        self.drop2 = nn.Dropout(0.3)
        self.proj = CentroidProjection(64, num_classes=10, alpha=PROJ_ALPHA, beta=PROJ_BETA, infer=False)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x, labels=None):
        x = self.drop1(torch.relu(self.bn1(self.fc1(x))))
        x = self.drop2(torch.relu(self.ln2(self.fc2(x))))
        x = self.proj(x, labels)
        x = self.fc3(x)
        return x

    def compute_centroids(self, loader, device):
        self.eval()
        sums = torch.zeros(10, 64, device=device)
        counts = torch.zeros(10, device=device)
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x = batch_x.to(device)
                h = torch.relu(self.ln2(self.fc2(self.drop1(torch.relu(self.bn1(self.fc1(batch_x)))))))
                for c in range(10):
                    mask = (batch_y == c)
                    if mask.any():
                        sums[c] += h[mask].sum(dim=0)
                        counts[c] += mask.sum()
        return sums / counts.unsqueeze(1).clamp(min=1)

# Model C: 09-style (projection during training AND inference)
class MLP_C(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.ln2 = nn.LayerNorm(64)
        self.drop2 = nn.Dropout(0.3)
        self.proj = CentroidProjection(64, num_classes=10, alpha=PROJ_ALPHA, beta=PROJ_BETA, infer=True)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x, labels=None):
        x = self.drop1(torch.relu(self.bn1(self.fc1(x))))
        x = self.drop2(torch.relu(self.ln2(self.fc2(x))))
        x = self.proj(x, labels)
        x = self.fc3(x)
        return x

    def compute_centroids(self, loader, device):
        self.eval()
        sums = torch.zeros(10, 64, device=device)
        counts = torch.zeros(10, device=device)
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x = batch_x.to(device)
                h = torch.relu(self.ln2(self.fc2(self.drop1(torch.relu(self.bn1(self.fc1(batch_x)))))))
                for c in range(10):
                    mask = (batch_y == c)
                    if mask.any():
                        sums[c] += h[mask].sum(dim=0)
                        counts[c] += mask.sum()
        return sums / counts.unsqueeze(1).clamp(min=1)

print("Model A (06-style, no projection):")
model_a = MLP_A().to(device)
print(model_a)
print()
print(f"Model B (07-style, projection train-only, alpha={PROJ_ALPHA}, beta={PROJ_BETA}):")
model_b = MLP_B().to(device)
print(model_b)
print()
print(f"Model C (09-style, projection train+infer, alpha={PROJ_ALPHA}, beta={PROJ_BETA}):")
model_c = MLP_C().to(device)
print(model_c)

Model A (06-style, no projection):
MLP_A(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

Model B (07-style, projection train-only, alpha=0.1, beta=0.03):
MLP_B(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (proj): CentroidProjection()
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

Model C (09-sty

## 3. Train Model A (06 config — no projection)

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer_a = optim.Adam(model_a.parameters(), lr=0.001, weight_decay=1e-4)
l1_lambda = 1e-4
num_epochs = 10

for epoch in range(num_epochs):
    model_a.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer_a.zero_grad()
        output = model_a(batch_x)
        l1_norm = sum(p.abs().sum() for p in model_a.parameters())
        loss = criterion(output, batch_y) + l1_lambda * l1_norm
        loss.backward()
        optimizer_a.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}: loss={running_loss / len(train_loader):.4f}")

# Final eval accuracy
model_a.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        correct += (model_a(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"\nModel A test accuracy: {correct/total:.4f}")

Epoch 0: loss=0.9867
Epoch 1: loss=0.4740
Epoch 2: loss=0.3865
Epoch 3: loss=0.3459
Epoch 4: loss=0.3245
Epoch 5: loss=0.3003
Epoch 6: loss=0.2948
Epoch 7: loss=0.2882
Epoch 8: loss=0.2738
Epoch 9: loss=0.2706

Model A test accuracy: 0.9754


## 4. Train Model B (07 config — with CentroidProjection)

In [14]:
# Reset seed so model B trains with same data order
torch.manual_seed(42)
model_b = MLP_B().to(device)
train_loader_b = DataLoader(train_dataset, batch_size=512, shuffle=True, generator=torch.Generator().manual_seed(0))

optimizer_b = optim.Adam(model_b.parameters(), lr=0.001, weight_decay=1e-4)

for epoch in range(num_epochs):
    centroids = model_b.compute_centroids(train_loader_b, device)
    model_b.proj.update_centroids(centroids)

    model_b.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader_b:
        optimizer_b.zero_grad()
        output = model_b(batch_x, labels=batch_y)
        l1_norm = sum(p.abs().sum() for p in model_b.parameters())
        loss = criterion(output, batch_y) + l1_lambda * l1_norm
        loss.backward()
        optimizer_b.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}: loss={running_loss / len(train_loader_b):.4f}")

# Final eval accuracy
model_b.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        correct += (model_b(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"\nModel B test accuracy: {correct/total:.4f}")

Epoch 0: loss=1.0266
Epoch 1: loss=0.3800
Epoch 2: loss=0.3021
Epoch 3: loss=0.2687
Epoch 4: loss=0.2488
Epoch 5: loss=0.2369
Epoch 6: loss=0.2264
Epoch 7: loss=0.2184
Epoch 8: loss=0.2092
Epoch 9: loss=0.2038

Model B test accuracy: 0.9728


## 5. Train Model C (09 config — projection at inference too)

In [15]:
# Reset seed so model C trains with same data order
torch.manual_seed(42)
model_c = MLP_C().to(device)
train_loader_c = DataLoader(train_dataset, batch_size=512, shuffle=True, generator=torch.Generator().manual_seed(0))

optimizer_c = optim.Adam(model_c.parameters(), lr=0.001, weight_decay=1e-4)

for epoch in range(num_epochs):
    centroids = model_c.compute_centroids(train_loader_c, device)
    model_c.proj.update_centroids(centroids)

    model_c.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader_c:
        optimizer_c.zero_grad()
        output = model_c(batch_x, labels=batch_y)
        l1_norm = sum(p.abs().sum() for p in model_c.parameters())
        loss = criterion(output, batch_y) + l1_lambda * l1_norm
        loss.backward()
        optimizer_c.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}: loss={running_loss / len(train_loader_c):.4f}")

# Final eval accuracy (projection active at inference via nearest-centroid)
model_c.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        correct += (model_c(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"\nModel C test accuracy: {correct/total:.4f}")

Epoch 0: loss=1.0266
Epoch 1: loss=0.3800
Epoch 2: loss=0.3021
Epoch 3: loss=0.2687
Epoch 4: loss=0.2488
Epoch 5: loss=0.2369
Epoch 6: loss=0.2264
Epoch 7: loss=0.2184
Epoch 8: loss=0.2092
Epoch 9: loss=0.2038

Model C test accuracy: 0.9717


## 6. Extract Hidden-Layer Activations

In [16]:
def extract_activations(model, loader, hook_layers):
    """Extract activations from named layers using forward hooks.
    
    Args:
        model: the model to extract from
        loader: data loader
        hook_layers: dict of {name: module} to hook
    
    Returns:
        dict of {name: np.array of shape [N, dim]}, plus labels array
    """
    model.eval()
    activations = {name: [] for name in hook_layers}
    all_labels = []
    hooks = []

    def make_hook(name):
        def hook_fn(module, input, output):
            activations[name].append(output.detach().cpu())
        return hook_fn

    for name, module in hook_layers.items():
        hooks.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        for x, y in loader:
            model(x)
            all_labels.append(y)

    for h in hooks:
        h.remove()

    result = {name: torch.cat(acts).numpy() for name, acts in activations.items()}
    labels = torch.cat(all_labels).numpy()
    return result, labels

# Model A: hidden_2 only
acts_a, labels_a = extract_activations(model_a, test_loader, {
    "hidden_2": model_a.drop2,
})

# Model B: hidden_2 + projected (projected is no-op at eval)
acts_b, labels_b = extract_activations(model_b, test_loader, {
    "hidden_2": model_b.drop2,
    "projected": model_b.proj,
})

# Model C: hidden_2 + projected (projected IS active at eval via nearest-centroid)
acts_c, labels_c = extract_activations(model_c, test_loader, {
    "hidden_2": model_c.drop2,
    "projected": model_c.proj,
})

print(f"Model A hidden_2:  {acts_a['hidden_2'].shape}")
print(f"Model B hidden_2:  {acts_b['hidden_2'].shape}")
print(f"Model B projected: {acts_b['projected'].shape}")
print(f"Model C hidden_2:  {acts_c['hidden_2'].shape}")
print(f"Model C projected: {acts_c['projected'].shape}")
print(f"Labels: {labels_a.shape}")

Model A hidden_2:  (10000, 64)
Model B hidden_2:  (10000, 64)
Model B projected: (10000, 64)
Model C hidden_2:  (10000, 64)
Model C projected: (10000, 64)
Labels: (10000,)


## 7. Logistic Regression + Silhouette Scores

In [17]:
from sklearn.model_selection import train_test_split

results = []

datasets_to_test = {
    "Model A (06) — hidden_2":     acts_a["hidden_2"],
    "Model B (07) — hidden_2":     acts_b["hidden_2"],
    "Model B (07) — projected":    acts_b["projected"],
    "Model C (09) — hidden_2":     acts_c["hidden_2"],
    "Model C (09) — projected":    acts_c["projected"],
}

for name, X in datasets_to_test.items():
    y = labels_a  # same test set for all

    # Train/test split for logistic regression
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # Logistic regression
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Silhouette score (subsample for speed)
    n_sil = min(5000, len(X))
    idx = np.random.RandomState(42).choice(len(X), n_sil, replace=False)
    sil = silhouette_score(X[idx], y[idx])

    results.append({"name": name, "logreg_acc": acc, "silhouette": sil})
    print(f"{name}:")
    print(f"  Logistic Regression accuracy: {acc:.4f}")
    print(f"  Silhouette score:             {sil:.4f}")
    print()

Model A (06) — hidden_2:
  Logistic Regression accuracy: 0.9777
  Silhouette score:             0.5432

Model B (07) — hidden_2:
  Logistic Regression accuracy: 0.9757
  Silhouette score:             0.5017

Model B (07) — projected:
  Logistic Regression accuracy: 0.9757
  Silhouette score:             0.5017

Model C (09) — hidden_2:
  Logistic Regression accuracy: 0.9757
  Silhouette score:             0.5017

Model C (09) — projected:
  Logistic Regression accuracy: 0.9743
  Silhouette score:             0.5438



## 8. Summary

In [18]:
print(f"{'Layer':<35} {'LogReg Acc':>10} {'Silhouette':>11}")
print("-" * 58)
for r in results:
    print(f"{r['name']:<35} {r['logreg_acc']:>10.4f} {r['silhouette']:>11.4f}")

print()
# Delta analysis
acc_a = results[0]["logreg_acc"]
sil_a = results[0]["silhouette"]

print("Deltas vs Model A (06) hidden_2 baseline:")
for r in results[1:]:
    d_acc = r["logreg_acc"] - acc_a
    d_sil = r["silhouette"] - sil_a
    print(f"  {r['name']:<33} LogReg {d_acc:+.4f}, Silhouette {d_sil:+.4f}")

print()
print("Projection effect (within each model):")
# B: hidden_2 vs projected (should be identical — no-op)
d_acc = results[2]["logreg_acc"] - results[1]["logreg_acc"]
d_sil = results[2]["silhouette"] - results[1]["silhouette"]
print(f"  Model B hidden_2 → projected:  LogReg {d_acc:+.4f}, Silhouette {d_sil:+.4f}  (no-op expected)")
# C: hidden_2 vs projected (the real test)
d_acc = results[4]["logreg_acc"] - results[3]["logreg_acc"]
d_sil = results[4]["silhouette"] - results[3]["silhouette"]
print(f"  Model C hidden_2 → projected:  LogReg {d_acc:+.4f}, Silhouette {d_sil:+.4f}  ← inference projection effect")

Layer                               LogReg Acc  Silhouette
----------------------------------------------------------
Model A (06) — hidden_2                 0.9777      0.5432
Model B (07) — hidden_2                 0.9757      0.5017
Model B (07) — projected                0.9757      0.5017
Model C (09) — hidden_2                 0.9757      0.5017
Model C (09) — projected                0.9743      0.5438

Deltas vs Model A (06) hidden_2 baseline:
  Model B (07) — hidden_2           LogReg -0.0020, Silhouette -0.0416
  Model B (07) — projected          LogReg -0.0020, Silhouette -0.0416
  Model C (09) — hidden_2           LogReg -0.0020, Silhouette -0.0416
  Model C (09) — projected          LogReg -0.0033, Silhouette +0.0006

Projection effect (within each model):
  Model B hidden_2 → projected:  LogReg +0.0000, Silhouette +0.0000  (no-op expected)
  Model C hidden_2 → projected:  LogReg -0.0013, Silhouette +0.0421  ← inference projection effect
